#메모

hidden layers\
early stopping\
free_bits 비활성화\
iterable data set\
ordinal cross entropy loss\
\
0720 : mcar = (50, 0.3)
0807 : mnar = (50, 0.3)




#PreProcess

In [49]:
import torch
import pandas as pd
import numpy as np

def load_data(file_path, chunk_size=10000):
    """Loads data from a large CSV file in chunks."""
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        yield chunk

def preprocess_chunk(chunk):
    """Processes a chunk of data: replace NaNs, create mask, calculate means, create conditional vector c."""
    # Ensure chunk is treated as a DataFrame if it's not already
    if not isinstance(chunk, pd.DataFrame):
         chunk = pd.DataFrame(chunk)

    # 1. Replace NaNs with 0
    chunk_filled = chunk.fillna(0)

    # 2. Create a binary mask and ensure float32 type
    # Work with .values to get NumPy array for mask operation
    mask_values = (~chunk.isna()).values.astype(np.float32)

    # 3. Calculate per-sample column means for observed values (kept for potential future use or debugging, but not used in c)
    # Use .values to avoid potential issues with index alignment during division
    observed_values = chunk.values * mask_values # mask_values is now np.float32
    sum_observed = np.nansum(observed_values, axis=1)
    num_observed = np.sum(mask_values, axis=1)
    # Avoid division by zero and ensure float32 type for means
    per_sample_means = np.divide(sum_observed, num_observed, out=np.zeros_like(sum_observed, dtype=np.float32), where=num_observed != 0)
    per_sample_means = per_sample_means[:, np.newaxis] # Reshape to (n_samples, 1)

    # 4. Create conditional vector c using only the mask and ensure float32 type
    # Ensure shapes are compatible for concatenation - use mask_values (NumPy array)
    # c = np.concatenate([mask_values, per_sample_means], axis=1).astype(np.float32) # Original line with mean
    c = mask_values.astype(np.float32) # Modified: c consists only of the mask


    # Return chunk_filled as DataFrame, mask as DataFrame (if needed later with index/columns), c as NumPy array
    # Let's return mask as a DataFrame for consistency if other functions expect it
    mask_df = pd.DataFrame(mask_values, index=chunk.index, columns=chunk.columns)

    return chunk_filled, mask_df, c

#CVAE

In [50]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Helper function for creating sequential layers
def create_sequential_layers(input_dim, hidden_dims, output_dim, activation=nn.ReLU()):
    """Creates a sequential model with hidden layers."""
    layers = []
    current_dim = input_dim
    for h_dim in hidden_dims:
        layers.append(nn.Linear(current_dim, h_dim))
        layers.append(activation)
        # Optionally add Dropout here: layers.append(nn.Dropout(p=0.1))
        current_dim = h_dim
    layers.append(nn.Linear(current_dim, output_dim))
    return nn.Sequential(*layers)


class Encoder(nn.Module):
    """Encoder module for CVAE."""
    # Encoder now receives embedded input
    def __init__(self, num_items, embedding_dim, c_dim, latent_dim, encoder_hidden_dims):
        super(Encoder, self).__init__()
        self.num_items = num_items
        self.embedding_dim = embedding_dim # Embedding dimension for each item
        self.c_dim = c_dim # Dimension of the conditional vector
        self.latent_dim = latent_dim
        self.encoder_hidden_dims = encoder_hidden_dims # List of hidden layer dimensions

        # Input dimension to the encoder is the flattened embedded item vectors + conditional vector
        # Each item is now embedding_dim, so total item input is num_items * embedding_dim
        input_dim = self.num_items * self.embedding_dim + self.c_dim

        # Define encoder layers dynamically based on encoder_hidden_dims
        self.encoder_layers = create_sequential_layers(input_dim, encoder_hidden_dims, 2 * latent_dim) # Output 2*latent_dim for mu and logvar


    def forward(self, x_embedded, c):
        """
        Args:
            x_embedded (Tensor): Embedded input data tensor (batch_size, num_items, embedding_dim).
            c (Tensor): Conditional vector tensor (batch_size, c_dim).
        Returns:
            Tuple[Tensor, Tensor]: mu (mean) and logvar (log variance) of the latent distribution.
        """
        # Flatten the embedded item vectors
        # Reshape x_embedded from (batch_size, num_items, embedding_dim) to (batch_size, num_items * embedding_dim)
        x_flattened = x_embedded.view(x_embedded.size(0), -1) # -1 infers the size (num_items * embedding_dim)

        # Concatenate flattened embedded input and conditional vector
        # Ensure dimensions are compatible for concatenation (batch_size, dim)
        if x_flattened.size(0) != c.size(0):
             raise ValueError(f"Batch size mismatch between flattened embedded input ({x_flattened.size(0)}) and conditional c ({c.size(0)})")

        # Ensure x_flattened and c are 2D tensors for concatenation along dimension 1
        if x_flattened.ndim != 2 or c.ndim != 2:
             raise ValueError(f"Flattened embedded input ({x_flattened.ndim}D) or conditional c ({c.ndim}D) is not 2D.")


        # Concatenate along the column dimension (dim=1)
        input_combined = torch.cat((x_flattened, c), dim=1) # Shape (batch_size, num_items * embedding_dim + c_dim)

        # Pass through the encoder layers
        encoded = self.encoder_layers(input_combined)

        # Split the output into mu and logvar
        mu = encoded[:, :self.latent_dim]
        logvar = encoded[:, self.latent_dim:]

        return mu, logvar


class Decoder(nn.Module):
    """Decoder module for CVAE."""
    # Decoder now reconstructs embeddings, then maps to logits
    def __init__(self, num_items, embedding_dim, c_dim, latent_dim, decoder_hidden_dims, num_ratings):
        super(Decoder, self).__init__()
        self.num_items = num_items
        self.embedding_dim = embedding_dim # Embedding dimension to reconstruct
        self.c_dim = c_dim # Dimension of the conditional vector
        self.latent_dim = latent_dim
        self.decoder_hidden_dims = decoder_hidden_dims # List of hidden layer dimensions
        self.num_ratings = num_ratings # Number of ordinal categories

        # Input dimension to the decoder is the latent vector + conditional vector
        input_dim = self.latent_dim + self.c_dim

        # Define decoder layers to reconstruct flattened embeddings
        # The output dimension should be num_items * embedding_dim (the flattened embedding space)
        self.decoder_layers = create_sequential_layers(input_dim, decoder_hidden_dims, num_items * embedding_dim) # Output num_items * embedding_dim

        # Final layer to map reconstructed embedding for each item to logits over num_ratings
        # Input dimension is embedding_dim (for a single item's reconstructed embedding)
        # Output dimension is num_ratings (logits for each category)
        self.final_layer = nn.Linear(embedding_dim, num_ratings)


    def forward(self, z, c):
        """
        Args:
            z (Tensor): Latent vector (batch_size, latent_dim).
            c (Tensor): Conditional vector tensor (batch_size, c_dim).
        Returns:
            Tensor: Logits for each item (batch_size, num_items, num_ratings).
        """
        # Concatenate latent vector and conditional vector
        # Ensure dimensions are compatible (batch_size, dim)
        if z.size(0) != c.size(0):
             raise ValueError(f"Batch size mismatch between latent z ({z.size(0)}) and conditional c ({c.size(0)})")

        if z.ndim != 2 or c.ndim != 2:
             raise ValueError(f"Latent z ({z.ndim}D) or conditional c ({c.ndim}D) is not 2D.")


        input_combined = torch.cat((z, c), dim=1) # Shape (batch_size, latent_dim + c_dim)

        # Pass through the decoder layers to reconstruct flattened embeddings
        reconstructed_embeddings_flattened = self.decoder_layers(input_combined) # Shape (batch_size, num_items * embedding_dim)

        # Reshape flattened embeddings to (batch_size * num_items, embedding_dim)
        # to apply the final layer item by item
        reconstructed_embeddings_per_item = reconstructed_embeddings_flattened.view(-1, self.embedding_dim) # Shape (batch_size * num_items, embedding_dim)

        # Apply the final layer to get logits for each item
        logits_per_item = self.final_layer(reconstructed_embeddings_per_item) # Shape (batch_size * num_items, num_ratings)

        # Reshape back to (batch_size, num_items, num_ratings)
        logits = logits_per_item.view(-1, self.num_items, self.num_ratings)

        return logits


class CVAE(nn.Module):
    """Conditional Variational Autoencoder for ordinal data imputation with embedding."""
    def __init__(self, num_items, embedding_dim, c_dim, latent_dim, encoder_hidden_dims, decoder_hidden_dims, num_ratings):
        super(CVAE, self).__init__()
        self.num_items = num_items
        self.embedding_dim = embedding_dim # Embedding dimension for each rating category
        self.c_dim = c_dim
        self.latent_dim = latent_dim
        self.encoder_hidden_dims = encoder_hidden_dims
        self.decoder_hidden_dims = decoder_hidden_dims
        self.num_ratings = num_ratings # Number of ordinal categories

        # Define the Ordinal Embedding layer
        # We need an embedding for each rating category (1 to num_ratings)
        # Plus potentially an embedding for the 'missing' or 'filled-in' value (e.g., 0)
        # Assuming filled missing values are represented as index 0, and ratings 1-5 as indices 1-5
        # So total number of embeddings needed is num_ratings + 1
        self.embedding = nn.Embedding(num_embeddings=num_ratings + 1, embedding_dim=embedding_dim)


        # Initialize Encoder and Decoder with hidden layers
        # Encoder receives flattened embedded input
        self.encoder = Encoder(num_items, embedding_dim, c_dim, latent_dim, encoder_hidden_dims)
        # Decoder reconstructs embeddings and maps to logits
        self.decoder = Decoder(num_items, embedding_dim, c_dim, latent_dim, decoder_hidden_dims, num_ratings)


    def forward(self, x, c):
        """
        Args:
            x (Tensor): Input data tensor (batch_size, num_items).
                       Assumes integer indices suitable for embedding (e.g., 0 for filled missing, 1-5 for ratings).
            c (Tensor): Conditional vector tensor (batch_size, c_dim). Mask + other info.
        Returns:
            Tuple[Tensor, Tensor, Tensor]: logits, mu, logvar.
        """
        # Apply the embedding layer to the input data x
        # x should contain integer indices (batch_size, num_items)
        x_embedded = self.embedding(x.long()) # Ensure x is long tensor for embedding layer
        # x_embedded shape is (batch_size, num_items, embedding_dim)

        # Encode the embedded input to get mu and logvar
        mu, logvar = self.encoder(x_embedded, c)

        # Reparameterization trick to sample z from the latent distribution
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std) # Sample from standard normal distribution
        z = mu + eps * std # Sample z

        # Decode the latent vector z to get logits
        logits = self.decoder(z, c) # Shape (batch_size, num_items, num_ratings)

        return logits, mu, logvar

# --- Loss Functions ---
# ordinal_cross_entropy_loss, kl_divergence_loss, combined_loss
# These should be defined in separate cells (e.g., cell ce812be1)

# Ensure NUM_RATINGS is defined in Hyperparameters cell (QT_6WTgCfUN3)
# It's needed for the Embedding layer and Decoder output.
# Ensure embedding_dim is defined in Hyperparameters cell (QT_6WTgCfUN3)

#LossFunction

In [51]:
import torch.nn.functional as F

def ordinal_cross_entropy_loss(logits, targets, mask):
    """
    Calculates the cross-entropy loss only for the observed values (1-5).

    Args:
        logits (torch.Tensor): Predicted logits (batch_size, num_items, num_classes).
        targets (torch.Tensor): True Likert values (batch_size, num_items).
        mask (torch.Tensor): Binary mask (batch_size, num_items), 1 for observed, 0 for missing.

    Returns:
        torch.Tensor: Scaled mean cross-entropy loss over observed values.
    """
    # Flatten tensors
    logits_flat = logits.view(-1, logits.size(-1)) # (batch_size * num_items, num_classes)
    targets_flat = targets.view(-1) # (batch_size * num_items)
    mask_flat = mask.view(-1) # (batch_size * num_items)

    # Create a combined mask: originally observed AND target value is within 1-5 range
    valid_observed_mask = (mask_flat == 1) & (targets_flat >= 1) & (targets_flat <= 5)

    # Apply combined mask: only consider valid observed values
    observed_logits = logits_flat[valid_observed_mask]
    observed_targets = targets_flat[valid_observed_mask]

    num_observed = observed_targets.size(0)

    if num_observed == 0:
        return torch.tensor(0.0, device=logits.device)

    # Ensure observed targets are long type and adjust from 1-5 to 0-4 for cross_entropy index
    # Now we are sure observed_targets are in [1, 5] range
    observed_targets = observed_targets.long() - 1

    # Calculate cross-entropy loss for observed items
    reconstruction_loss = F.cross_entropy(observed_logits, observed_targets, reduction='mean')

    return reconstruction_loss

def kl_divergence_loss(mu, logvar, free_bits): # Added free_bits parameter
    """
    Calculates the KL divergence loss with optional free bits.

    Args:
        mu (torch.Tensor): Mean of the latent distribution.
        logvar (torch.Tensor): Log variance of the latent distribution.
        free_bits (float): The free bits threshold.

    Returns:
        torch.Tensor: KL divergence loss.
    """
    # KL divergence between N(mu, var) and N(0, 1)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    # Apply free bits: the KL loss is not penalized if it's below free_bits
    kl_loss = torch.max(kl_loss, torch.tensor(free_bits * mu.size(0), device=mu.device)) # Scale free_bits by batch size

    return kl_loss

def combined_loss(reconstruction_loss, kl_loss, beta):
    """
    Combines reconstruction loss and KL divergence loss with beta weighting.

    Args:
        reconstruction_loss (torch.Tensor): The reconstruction loss.
        kl_loss (torch.Tensor): The KL divergence loss.
        beta (float): The weight for the KL divergence loss.

    Returns:
        torch.Tensor: The total loss.
    """
    return reconstruction_loss + beta * kl_loss

#CalculateMetrics

In [52]:
def calculate_metrics(logits, targets, mask):
    """
    Calculates Accuracy, MAE, and RMSE for observed values.

    Args:
        logits (torch.Tensor): Predicted logits (batch_size, num_items, num_classes).
        targets (torch.Tensor): True Likert values (batch_size, num_items).
        mask (torch.Tensor): Binary mask (batch_size, num_items).

    Returns:
        tuple: (accuracy, mae, rmse)
    """
    # Get predicted Likert values (1-5) from logits
    # Find the class with the highest logit for each item
    predicted_classes = torch.argmax(logits, dim=-1) # (batch_size, num_items)
    predicted_values = predicted_classes + 1 # Adjust predicted classes from 0-4 to 1-5

    # Flatten tensors and apply mask
    predicted_flat = predicted_values.view(-1)
    targets_flat = targets.view(-1)
    mask_flat = mask.view(-1)

    # Apply mask to keep only observed values
    observed_predicted = predicted_flat[mask_flat == 1]
    observed_targets = targets_flat[mask_flat == 1]

    num_observed = observed_targets.size(0)

    if num_observed == 0:
        return 0.0, 0.0, 0.0 # Return 0 if no observed values

    # Accuracy: percentage of correctly predicted values among observed
    correct_predictions = (observed_predicted == observed_targets).sum().item()
    accuracy = correct_predictions / num_observed

    # MAE: Mean Absolute Error among observed
    mae = torch.abs(observed_predicted.float() - observed_targets.float()).mean().item()

    # RMSE: Root Mean Squared Error among observed
    rmse = torch.sqrt(torch.mean((observed_predicted.float() - observed_targets.float())**2)).item()

    return accuracy, mae, rmse

def apply_artificial_masking(data, mask, masking_prob):
    """
    Applies artificial masking to the data for evaluation purposes.

    Args:
        data (torch.Tensor): Input data (batch_size, num_items).
        mask (torch.Tensor): Original mask (batch_size, num_items).
        masking_prob (float): Probability of masking an originally observed value.

    Returns:
        tuple: (artificially_masked_data, artificial_mask)
    """
    # Create a new mask starting with the original mask
    artificial_mask = mask.clone()

    # Identify originally observed values
    observed_indices = (mask == 1)

    # Create a random mask for observed values based on masking_prob
    random_mask = torch.rand_like(data, dtype=torch.float) < masking_prob

    # Combine the random mask with the observed indices to only mask originally observed values
    mask_to_apply = observed_indices * random_mask

    # Apply artificial masking: set masked values to 0 in the data
    artificially_masked_data = data.clone()
    artificially_masked_data[mask_to_apply] = 0

    # Update the artificial mask: set masked positions to 0
    artificial_mask[mask_to_apply] = 0

    return artificially_masked_data, artificial_mask

# Placeholder for training loop, warm-up, and early stopping which will be implemented next
# Need to define model, optimizer, dataloaders, and hyperparameters first.
# This part of the code block only defines the necessary functions.

#HyperParameters

In [53]:
from google.colab import drive
drive.mount('/content/drive')

# 데이터 설정
NUM_RATINGS = 5
FILE_PATH = "/content/drive/MyDrive/Projects/CVAE_MI/big5_ans_only.csv" # 학습 데이터 파일 경로
CHUNK_SIZE = 10000 # 데이터 로딩 시 사용할 청크 크기

# 모델 설정
EMBEDDING_DIM = 2 # 아이템 임베딩 차원
LATENT_DIM = 8 # 잠재 공간 차원
NUM_ITEMS = 50 # Likert 척도 데이터의 아이템 수 (컬럼 수)
# C_DIM = NUM_ITEMS + 1 # 조건 벡터 c의 차원 (마스크 크기 + 평균 1) # Original line
C_DIM = NUM_ITEMS # 조건 벡터 c의 차원 (마스크 크기만) # Modified


# 인코더/디코더 은닉층 설정
ENCODER_HIDDEN_DIMS = [256, 128] # 인코더 은닉층 차원 리스트
DECODER_HIDDEN_DIMS = [32, 32] # 디코더 은닉층 차원 리스트


# 학습 설정
BATCH_SIZE = 128 # 학습 배치 크기
LEARNING_RATE = 1e-4 # Adam 옵티마이저 학습률
NUM_EPOCHS = 100 # 전체 학습 에폭 수

# VAE Loss 설정
FREE_BITS = 0.000001

# Beta 스케줄링 설정 (Linear Warmup)
BETA_FIX_EPOCHS = 10
BETA_START = 0.0001 # 베타의 최소값 (웜업 시작 값)
BETA_END = 0.001   # 베타의 최대값 (웜업 종료 값 및 이후 값)
BETA_WARMUP_EPOCHS = 50 # 베타 선형 웜업 기간 (에폭 단위)

# 평가 및 조기 종료 설정
EARLY_STOPPING_PATIENCE = 10 # 검증 손실 개선이 없을 때 조기 종료까지 기다릴 에폭 수
EARLY_STOPPING_START_EPOCH = 50 # 조기 종료 체크를 시작할 에폭 (0부터 시작)
MASKING_PROB_EVAL = 0.1 # 평가 시 인위적 결측을 생성할 확률 (원래 관측된 값에 대해)


# --- 인위적 결측 생성 설정 ---
# 학습 데이터셋에 인위적 결측을 생성하기 위한 설정입니다.
MISSINGNESS_CONFIG = {
    'mcar': (0, 0.30),        # (컬럼 개수, 해당 메커니즘으로 만들 총 결측률 목표)
    'mnar': (50, 0.30),        # (컬럼 개수, 해당 메커니즘으로 만들 총 결측률 목표)
    'mar': (0, 0.30),         # (컬럼 개수, 해당 메커니즘으로 만들 총 결측률 목표)
    'not_missing_count': 0, # NotMissing 컬럼 개수
    'random_allocation': True, # True: 무작위 할당, False: 사용자 지정
    # random_allocation=False 일 때 사용할 사용자 지정 컬럼 리스트 (예시)
    'manual_columns': {
        'mcar': ['Q1', 'Q5', 'Q10', 'Q15', 'Q20', 'Q25', 'Q30', 'Q35', 'Q40', 'Q45'],
        'mnar': ['Q2', 'Q6', 'Q11', 'Q16', 'Q21', 'Q26', 'Q31', 'Q36', 'Q41', 'Q46',
                 'Q3', 'Q7', 'Q12', 'Q17', 'Q22', 'Q27', 'Q32', 'Q37', 'Q42', 'Q47'],
        'mar': ['Q4', 'Q8', 'Q13', 'Q18', 'Q23', 'Q28', 'Q33', 'Q38', 'Q43', 'Q48'],
        'not_missing': ['Q9', 'Q14', 'Q19', 'Q24', 'Q29', 'Q34', 'Q39', 'Q44', 'Q49', 'Q50']
    }
}

# 데이터 파일에서 컬럼 이름을 미리 로드 (헤더만 읽기)
# 이 부분은 MISSINGNESS_CONFIG 검증에 필요하므로 여기에 두는 것이 편리합니다.
try:
    # Use the training file path defined above
    if 'FILE_PATH' not in globals():
        raise NameError("FILE_PATH 변수가 정의되지 않았습니다. 경로를 확인하세요.")
    # Read only the header row to get column names
    df_header = pd.read_csv(FILE_PATH, nrows=0)
    all_columns = df_header.columns.tolist()
    NUM_ITEMS_DATA = len(all_columns) # 실제 데이터의 컬럼 수
    print(f"데이터 파일에서 로드된 총 컬럼 수: {NUM_ITEMS_DATA}")
    print(f"컬럼 이름: {all_columns}") # Optional: print column names

    # 하이퍼파라미터 셀의 NUM_ITEMS와 데이터의 컬럼 수 일치 여부 확인 (경고)
    if 'NUM_ITEMS' not in globals():
         print("경고: 하이퍼파라미터 NUM_ITEMS가 정의되지 않았습니다. 데이터 컬럼 수로 설정합니다.")
         NUM_ITEMS = NUM_ITEMS_DATA # Set NUM_ITEMS if not defined
    elif NUM_ITEMS != NUM_ITEMS_DATA:
         print(f"경고: 하이퍼파라미터 NUM_ITEMS ({NUM_ITEMS})와 데이터 파일의 컬럼 수 ({NUM_ITEMS_DATA})가 다릅니다. 데이터 파일 기준으로 진행합니다.")
         NUM_ITEMS = NUM_ITEMS_DATA # Use data columns count


except FileNotFoundError:
    print(f"오류: 학습 데이터 파일을 찾을 수 없습니다. 경로를 확인하세요: {FILE_PATH}")
    # Exit or handle the error appropriately
    raise
except Exception as e:
    print(f"데이터 파일 헤더를 읽는 중 오류가 발생했습니다: {e}")
    raise


# 입력 파라미터 유효성 검사 (MISSINGNESS_CONFIG 기준)
total_config_cols = MISSINGNESS_CONFIG['mcar'][0] + MISSINGNESS_CONFIG['mnar'][0] + \
                      MISSINGNESS_CONFIG['mar'][0] + MISSINGNESS_CONFIG['not_missing_count']

if total_config_cols != NUM_ITEMS_DATA:
    raise ValueError(f"MISSINGNESS_CONFIG에 정의된 컬럼 개수 합({total_config_cols})이 데이터의 총 컬럼 수({NUM_ITEMS_DATA})와 일치하지 않습니다.")

# MAR 컬럼 개수와 NotMissing 컬럼 개수 일치 확인
if MISSINGNESS_CONFIG['mar'][0] != MISSINGNESS_CONFIG['not_missing_count']:
    raise ValueError(f"MAR 컬럼 개수({MISSINGNESS_CONFIG['mar'][0]})와 NotMissing 컬럼 개수({MISSINGNESS_CONFIG['not_missing_count']})가 일치해야 합니다.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
데이터 파일에서 로드된 총 컬럼 수: 50
컬럼 이름: ['EXT1', 'EXT2', 'EXT3', 'EXT4', 'EXT5', 'EXT6', 'EXT7', 'EXT8', 'EXT9', 'EXT10', 'EST1', 'EST2', 'EST3', 'EST4', 'EST5', 'EST6', 'EST7', 'EST8', 'EST9', 'EST10', 'AGR1', 'AGR2', 'AGR3', 'AGR4', 'AGR5', 'AGR6', 'AGR7', 'AGR8', 'AGR9', 'AGR10', 'CSN1', 'CSN2', 'CSN3', 'CSN4', 'CSN5', 'CSN6', 'CSN7', 'CSN8', 'CSN9', 'CSN10', 'OPN1', 'OPN2', 'OPN3', 'OPN4', 'OPN5', 'OPN6', 'OPN7', 'OPN8', 'OPN9', 'OPN10']


#IntroduceMissingValues

In [54]:
import pandas as pd
import numpy as np
import torch
import random # Import random for shuffling

# --- 1. 사용자 입력 파라미터 정의 및 검증 ---
# 하이퍼파라미터 셀(QT_6WTgCfUN3)에서 이미 정의하고 검증했습니다.
# MISSINGNESS_CONFIG, FILE_PATH, all_columns, NUM_ITEMS_DATA, NUM_ITEMS 변수를 사용합니다.


# --- 2. 컬럼 할당 로직 구현 ---

mcar_cols = []
mnar_cols = []
mar_cols = []
not_missing_cols = []
mar_not_missing_pairs = {} # MAR 컬럼과 대응되는 NotMissing 컬럼 매핑

if MISSINGNESS_CONFIG['random_allocation']:
    print("컬럼을 무작위로 할당합니다.")
    # 전체 컬럼 목록을 섞습니다.
    shuffled_cols = all_columns.copy()
    random.shuffle(shuffled_cols)

    # 설정된 개수만큼 각 그룹에 할당
    current_idx = 0
    mcar_cols = shuffled_cols[current_idx : current_idx + MISSINGNESS_CONFIG['mcar'][0]]
    current_idx += MISSINGNESS_CONFIG['mcar'][0]

    mnar_cols = shuffled_cols[current_idx : current_idx + MISSINGNESS_CONFIG['mnar'][0]]
    current_idx += MISSINGNESS_CONFIG['mnar'][0]

    mar_cols = shuffled_cols[current_idx : current_idx + MISSINGNESS_CONFIG['mar'][0]]
    current_idx += MISSINGNESS_CONFIG['mar'][0]

    not_missing_cols = shuffled_cols[current_idx : current_idx + MISSINGNESS_CONFIG['not_missing_count']]

    # MAR 컬럼과 NotMissing 컬럼을 무작위로 1:1 매핑
    shuffled_mar_cols = mar_cols.copy()
    shuffled_not_missing_cols = not_missing_cols.copy()
    random.shuffle(shuffled_mar_cols)
    random.shuffle(shuffled_not_missing_cols)
    mar_not_missing_pairs = {mar_col: nm_col for mar_col, nm_col in zip(shuffled_mar_cols, shuffled_not_missing_cols)}


else: # manual_allocation
    print("사용자 지정 컬럼을 할당합니다.")
    manual_cols_dict = MISSINGNESS_CONFIG.get('manual_columns')
    if not manual_cols_dict:
         raise ValueError("random_allocation이 False일 때 manual_columns 설정이 필요합니다.")

    mcar_cols = manual_cols_dict.get('mcar', [])
    mnar_cols = manual_cols_dict.get('mnar', [])
    mar_cols = manual_cols_dict.get('mar', [])
    not_missing_cols = manual_cols_dict.get('not_missing', [])

    # 사용자 지정 컬럼 목록의 유효성 및 개수 확인
    assigned_cols = set(mcar_cols + mnar_cols + mar_cols + not_missing_cols)
    total_config_cols = MISSINGNESS_CONFIG['mcar'][0] + MISSINGNESS_CONFIG['mnar'][0] + \
                      MISSINGNESS_CONFIG['mar'][0] + MISSINGNESS_CONFIG['not_missing_count']

    if len(assigned_cols) != NUM_ITEMS_DATA or len(assigned_cols) != total_config_cols:
         raise ValueError("사용자 지정 컬럼 목록의 총 개수가 데이터 컬럼 수와 일치하지 않거나 중복된 컬럼이 있습니다.")

    if len(mcar_cols) != MISSINGNESS_CONFIG['mcar'][0] or \
       len(mnar_cols) != MISSINGNESS_CONFIG['mnar'][0] or \
       len(mar_cols) != MISSINGNESS_CONFIG['mar'][0] or \
       len(not_missing_cols) != MISSINGNESS_CONFIG['not_missing_count']:
         raise ValueError("사용자 지정 컬럼 목록의 개수가 MISSINGNESS_CONFIG에 설정된 개수와 일치하지 않습니다.")

     # MAR 컬럼과 NotMissing 컬럼을 순서대로 1:1 매핑 (사용자 지정 시 순서 기반)
    if len(mar_cols) != len(not_missing_cols):
         raise ValueError("사용자 지정 시 MAR 컬럼 리스트와 NotMissing 컬럼 리스트의 길이가 일치해야 합니다.")
    mar_not_missing_pairs = {mar_col: nm_col for mar_col, nm_col in zip(mar_cols, not_missing_cols)}


print(f"\n할당 결과:")
print(f"MCAR 컬럼 ({len(mcar_cols)}개): {mcar_cols}")
print(f"MNAR 컬럼 ({len(mnar_cols)}개): {mnar_cols}")
print(f"MAR 컬럼 ({len(mar_cols)}개): {mar_cols}")
print(f"NotMissing 컬럼 ({len(not_missing_cols)}개): {not_missing_cols}")
print(f"MAR-NotMissing 매핑: {mar_not_missing_pairs}")

# 각 그룹별 목표 총 결측 비율
mcar_total_missing_rate = MISSINGNESS_CONFIG['mcar'][1]
mnar_total_missing_rate = MISSINGNESS_CONFIG['mnar'][1]
mar_total_missing_rate = MISSINGNESS_CONFIG['mar'][1]

# 이 할당 결과 (mcar_cols, mnar_cols, mar_cols, not_missing_cols, mar_not_missing_pairs)를
# LikertDataset 인스턴스에 전달해야 합니다.


# --- 3. 인위적 결측 생성 함수 (청크 단위 적용) ---

def introduce_missing_in_chunk(chunk_df, mcar_cols, mnar_cols, mar_not_missing_pairs,
                               mcar_total_missing_rate, mnar_total_missing_rate, mar_total_missing_rate):
    """
    데이터 청크에 인위적인 결측치 (NaN)를 생성합니다.
    이 함수는 LikertDataset.__iter__ 내부에서 호출됩니다.

    Args:
        chunk_df (pd.DataFrame): 현재 데이터 청크.
        mcar_cols (list): MCAR 결측을 적용할 컬럼 이름 리스트.
        mnar_cols (list): MNAR 결측을 적용할 컬럼 이름 리스트.
        mar_not_missing_pairs (dict): MAR 컬럼과 대응되는 NotMissing 컬럼의 매핑 딕셔너리.
        mcar_total_missing_rate (float): MCAR 그룹 전체의 총 결측률 목표 (청크 기준 아님, 확률 계산에 활용).
        mnar_total_missing_rate (float): MNAR 그룹 전체의 총 결측률 목표 (청크 기준 아님, 확률 계산에 활용).
        mar_total_missing_rate (float): MAR 그룹 전체의 총 결측률 목표 (청크 기준 아님, 확률 계산에 활용).

    Returns:
        pd.DataFrame: 인위적 결측치가 포함된 데이터 청크 (NaN으로 표시).
    """
    chunk_with_missing = chunk_df.copy()
    # --- Refactoring: Reset index to use position-based indexing safely ---
    # Store original index if needed later (not used for now)
    chunk_with_missing = chunk_with_missing.reset_index(drop=True)
    # --- End Refactoring ---

    num_rows = len(chunk_with_missing)
    all_cols_in_chunk = chunk_with_missing.columns # Get column names from the chunk


    # MCAR 결측 생성
    if mcar_cols:
        mcar_cols_in_chunk = [col for col in mcar_cols if col in all_cols_in_chunk]
        num_mcar_cols_in_chunk = len(mcar_cols_in_chunk)

        if num_mcar_cols_in_chunk > 0:
             prob_per_mcar_cell_in_group = mcar_total_missing_rate

             for col in mcar_cols_in_chunk:
                 # Only consider originally observed values for masking using the *reset* index
                 original_observed_indices_bool = chunk_with_missing[col].notna()
                 original_observed_indices = original_observed_indices_bool[original_observed_indices_bool].index # Get reset integer indices
                 num_original_observed = len(original_observed_indices)


                 if num_original_observed > 0:
                     mask_candidates = original_observed_indices # Candidates are the reset integer indices
                     # Apply probability to originally observed cells to get target count
                     num_to_mask = int(num_original_observed * prob_per_mcar_cell_in_group)

                     # --- Safety check for np.random.choice ---
                     # Ensure num_to_mask is positive and there are candidates to sample from
                     if num_to_mask > 0 and len(mask_candidates) > 0:
                         # Ensure num_to_mask does not exceed available candidates
                         num_to_mask_actual = min(num_to_mask, len(mask_candidates))
                         # Sample indices from the reset integer indices
                         indices_to_mask = np.random.choice(mask_candidates, num_to_mask_actual, replace=False)
                         # Apply masking using the reset integer indices (.loc works with integer index after reset)
                         chunk_with_missing.loc[indices_to_mask, col] = np.nan
                     # --- End Safety check ---


    # MNAR 결측 생성
    if mnar_cols:
        mnar_cols_in_chunk = [col for col in mnar_cols if col in all_cols_in_chunk]
        num_mnar_cols_in_chunk = len(mnar_cols_in_chunk)

        if num_mnar_cols_in_chunk > 0:
             prob_per_mnar_cell_in_group = mnar_total_missing_rate

             for col in mnar_cols_in_chunk:
                 # Only consider originally observed values using the *reset* index
                 original_observed_indices_bool = chunk_with_missing[col].notna()
                 original_observed_indices = original_observed_indices_bool[original_observed_indices_bool].index # Get reset integer indices
                 num_original_observed = len(original_observed_indices)

                 if num_original_observed > 0:
                    # Get observed values along with their *reset integer indices*
                    observed_series = chunk_with_missing.loc[original_observed_indices, col]

                    # MNAR: 값이 작을수록 결측 확률 높게 (분포와 반대 방향)
                    # Target missing count (based on approx total rate for the group)
                    target_missing_count_in_chunk_col = int(num_original_observed * (mnar_total_missing_rate / len(mnar_cols))) if len(mnar_cols) > 0 else 0

                    # --- Safety check and Index selection for np.random.choice ---
                    # Ensure target_missing_count_in_chunk_col is positive and there are original observed values
                    if target_missing_count_in_chunk_col > 0 and num_original_observed > 0:
                         # Find the threshold value corresponding to the lowest target_missing_count_in_chunk_col values
                         # Sort the observed series by value, keeping the reset integer index
                         sorted_observed = observed_series.sort_values(ascending=True)

                         # Ensure target_missing_count_in_chunk_col does not exceed number of available values
                         num_to_mask_actual = min(target_missing_count_in_chunk_col, len(sorted_observed))

                         if num_to_mask_actual > 0:
                             # Get the *reset integer indices* of the lowest `num_to_mask_actual` values
                             indices_to_mask = sorted_observed.head(num_to_mask_actual).index

                             # Ensure the list of indices to sample from is not empty before calling choice
                             if len(indices_to_mask) > 0:
                                  # Apply masking using the reset integer indices
                                  chunk_with_missing.loc[indices_to_mask, col] = np.nan
                             # else: # This case indicates an issue if num_to_mask_actual > 0
                             #     print(f"Warning: No indices found for MNAR masking in {col} despite target count {num_to_mask_actual}. sorted_observed length: {len(sorted_observed)}")

                    # --- End Safety check and Index selection ---


    # MAR 결측 생성
    if mar_not_missing_pairs:
        mar_cols_in_chunk = [col for col in mar_not_missing_pairs.keys() if col in all_cols_in_chunk]
        num_mar_cols_in_chunk = len(mar_cols_in_chunk)

        if num_mar_cols_in_chunk > 0:
             prob_per_mar_cell_in_group = mar_total_missing_rate

             for mar_col in mar_cols_in_chunk:
                 nm_col = mar_not_missing_pairs[mar_col]
                 if nm_col not in all_cols_in_chunk:
                     continue

                 # Consider only originally observed values in the MAR column using the *reset* index for filtering
                 original_observed_mar_indices_bool = chunk_with_missing[mar_col].notna()
                 original_observed_mar_indices = original_observed_mar_indices_bool[original_observed_mar_indices_bool].index # Get reset integer indices
                 num_original_observed_mar = len(original_observed_mar_indices)

                 if num_original_observed_mar > 0:
                      # Get the corresponding NotMissing values for these observed MAR indices using the *reset* index
                      corresponding_nm_values = chunk_with_missing.loc[original_observed_mar_indices, nm_col]

                      # Ensure corresponding_nm_values are not NaN if nm_col is NotMissing using the *reset* index
                      valid_mar_nm_indices_bool = corresponding_nm_values.notna()
                      # Filter original observed MAR indices based on valid NM values using the *reset* index
                      original_observed_mar_indices_filtered = original_observed_mar_indices[valid_mar_nm_indices_bool[original_observed_mar_indices].values] # Filter indices based on boolean series aligned by index
                      corresponding_nm_values_filtered = corresponding_nm_values[valid_mar_nm_indices_bool] # Filter the NM values series
                      num_original_observed_mar_filtered = len(original_observed_mar_indices_filtered)


                      if num_original_observed_mar_filtered == 0:
                           continue # No valid pairs left in this chunk


                      # MAR: NotMissing 컬럼 값이 높을수록 MAR 컬럼에 결측 확률 높게
                      # Target missing count (based on approx total rate for the group)
                      target_missing_count_in_chunk_col = int(num_original_observed_mar_filtered * (mar_total_missing_rate / len(mar_not_missing_pairs))) if len(mar_not_missing_pairs) > 0 else 0

                      # --- Safety check and Index selection for np.random.choice ---
                      # Ensure target_missing_count_in_chunk_col is positive and there are filtered observed values
                      if target_missing_count_in_chunk_col > 0 and num_original_observed_mar_filtered > 0:
                         # Find the threshold value in the *NotMissing* column corresponding to the
                         # target_missing_count_in_chunk_col values with the *highest* values in the NotMissing column.
                         # Sort the filtered NotMissing values by value, keeping the *reset integer index*
                         sorted_nm_values = corresponding_nm_values_filtered.sort_values(ascending=False) # Sort descending

                         # Ensure target_missing_count_in_chunk_col does not exceed number of available values
                         num_to_mask_actual = min(target_missing_count_in_chunk_col, len(sorted_nm_values))


                         if num_to_mask_actual > 0:
                              # Get the *reset integer indices* of the highest `num_to_mask_actual` values in the NM column
                              indices_to_mask = sorted_nm_values.head(num_to_mask_actual).index # These are reset integer indices

                              # Ensure the list of indices to sample from is not empty before calling choice
                              if len(indices_to_mask) > 0:
                                   # Apply masking using the reset integer indices
                                   chunk_with_missing.loc[indices_to_mask, mar_col] = np.nan
                              # else: # This case indicates an issue if num_to_mask_actual > 0
                              #     print(f"Warning: No indices found for MAR masking in {mar_col} despite target count {num_to_mask_actual}. sorted_nm_values length: {len(sorted_nm_values)}")
                      # --- End Safety check and Index selection ---


    # --- Refactoring: Restore original index if needed (not strictly required here) ---
    # chunk_with_missing.index = original_index # Only if the calling function expects original index
    # --- End Refactoring ---

    return chunk_with_missing

컬럼을 무작위로 할당합니다.

할당 결과:
MCAR 컬럼 (0개): []
MNAR 컬럼 (50개): ['EST8', 'AGR2', 'EXT5', 'CSN2', 'EST1', 'EST6', 'EST4', 'EXT9', 'OPN4', 'CSN6', 'AGR3', 'EXT4', 'OPN5', 'AGR5', 'OPN8', 'EST2', 'OPN9', 'AGR9', 'OPN6', 'EXT7', 'EXT3', 'AGR10', 'EST7', 'EST9', 'OPN1', 'AGR7', 'CSN7', 'OPN10', 'EXT10', 'EST10', 'CSN1', 'AGR6', 'CSN10', 'EXT8', 'CSN9', 'AGR4', 'CSN4', 'AGR1', 'CSN8', 'OPN3', 'AGR8', 'CSN5', 'CSN3', 'EXT2', 'OPN7', 'EXT6', 'EST5', 'EST3', 'OPN2', 'EXT1']
MAR 컬럼 (0개): []
NotMissing 컬럼 (0개): []
MAR-NotMissing 매핑: {}


#LikertDataset

In [55]:


class LikertDataset(torch.utils.data.IterableDataset):
    """Custom PyTorch IterableDataset for Likert scale data with artificial missingness."""
    def __init__(self, file_path, chunk_size=10000, all_columns=[],
                 mcar_cols=[], mnar_cols=[], mar_not_missing_pairs={},
                 mcar_total_missing_rate=0.0, mnar_total_missing_rate=0.0, mar_total_missing_rate=0.0):
        super().__init__()
        self.file_path = file_path
        self.chunk_size = chunk_size
        self.all_columns = all_columns # Store column names
        # Store missingness configuration
        self.mcar_cols = mcar_cols
        self.mnar_cols = mnar_cols
        self.mar_not_missing_pairs = mar_not_missing_pairs
        self.mcar_total_missing_rate = mcar_total_missing_rate
        self.mnar_total_missing_rate = mnar_total_missing_rate
        self.mar_total_missing_rate = mar_total_missing_rate

        self.total_rows = self._get_total_rows() # Calculate and cache total rows in init
        print(f"Dataset initialized with {self.total_rows} total rows (approx).")
        print(f"Missingness config: MCAR({len(self.mcar_cols)}, {self.mcar_total_missing_rate}), MNAR({len(self.mnar_cols)}, {self.mnar_total_missing_rate}), MAR({len(self.mar_not_missing_pairs)}, {self.mar_total_missing_rate})")


    def _get_total_rows(self):
        """Helper to get total rows (optional, can be slow for very large files)."""
        # Re-use the logic from the original cell 95d21f6e
        try:
            # Use 'rb' mode for binary read to handle different line endings
            with open(self.file_path, 'rb') as f:
                 # Iterate through lines and count, subtracting header
                 total_rows = sum(1 for _ in f) - 1
            return total_rows
        except Exception as e:
            print(f"Warning: Could not determine total rows for chunk shuffling: {e}")
            return 0 # Assume 0 if cannot determine


    def __iter__(self):
        # Get worker info if using multiple workers in DataLoader
        worker_info = torch.utils.data.get_worker_info()
        total_rows = self.total_rows # Refer to the cached total rows
        num_chunks = (total_rows + self.chunk_size - 1) // self.chunk_size # Calculate total number of chunks

        if worker_info is None:  # Single-process data loading
            start_chunk = 0
            end_chunk = num_chunks
            # Generate a random order of chunks for the single worker
            chunk_order = np.random.permutation(range(start_chunk, end_chunk))

        else:  # Multiple workers
            # Distribute chunks among workers
            per_worker_chunks = int(np.ceil(num_chunks / float(worker_info.num_workers)))
            worker_id = worker_info.id
            start_chunk = worker_id * per_worker_chunks
            end_chunk = min(start_chunk + per_worker_chunks, num_chunks)
            # Generate a random order of chunks for this worker
            chunk_order = np.random.permutation(range(start_chunk, end_chunk))


        # Iterate through data chunks in the random order
        for chunk_idx in chunk_order:
            # Calculate the actual start row for this chunk
            start_row = chunk_idx * self.chunk_size
            # Use pandas read_csv with skiprows and nrows to read a specific chunk
            try:
                # Add +1 to skiprows to account for the header row, and read nrows=chunk_size
                # Need column names for missingness generation and preprocess_chunk
                # Use the stored column names
                if not self.all_columns:
                     # This check should ideally not be reached if all_columns was passed in __init__
                     # and is not empty. If it is reached, it means self.all_columns is empty.
                     raise ValueError("Column names not provided to LikertDataset (self.all_columns is empty).")

                chunk = pd.read_csv(self.file_path,
                                    skiprows=start_row + 1, # Skip header + previous rows
                                    nrows=self.chunk_size,
                                    header=None, # No header in the chunk itself
                                    names=self.all_columns) # Assign column names


                if chunk.empty:
                     continue # Skip empty chunks

                # --- Introduce Artificial Missingness in Chunk ---
                # Only apply missingness if the dataset is configured for it
                if self.mcar_cols or self.mnar_cols or self.mar_not_missing_pairs:
                     # Pass the loaded chunk and the missingness configuration stored in self
                     chunk_with_missing = introduce_missing_in_chunk(
                         chunk,
                         self.mcar_cols,
                         self.mnar_cols,
                         self.mar_not_missing_pairs,
                         self.mcar_total_missing_rate,
                         self.mnar_total_missing_rate,
                         self.mar_total_missing_rate
                     )
                else:
                    # If no missingness configured, use the original chunk
                    chunk_with_missing = chunk.copy()


                # --- Preprocess the Chunk (with potential missingness) ---
                # preprocess_chunk expects a DataFrame with potential NaNs (from introduce_missing_in_chunk)
                # It handles NaNs (fills with 0) and creates the mask and conditional vector c.
                # Ensure preprocess_chunk is defined in another cell (e.g., ce812be1)
                if 'preprocess_chunk' not in globals() or not callable(preprocess_chunk):
                    raise NameError("preprocess_chunk function is not defined. Please ensure the cell defining it has been executed.")

                chunk_filled, mask_df, c = preprocess_chunk(chunk_with_missing)

                # Shuffle the indices within the chunk
                sample_indices = np.random.permutation(len(chunk_filled))

                # Yield each sample from the processed chunk in the shuffled order
                # Ensure chunk_filled, mask_df, c are aligned by index before iterating sample_indices
                # Use .iloc[j] after applying sample_indices permutation to the index
                chunk_filled_shuffled = chunk_filled.iloc[sample_indices]
                mask_df_shuffled = mask_df.iloc[sample_indices]
                c_shuffled = c[sample_indices] # c is a numpy array

                for j in range(len(chunk_filled_shuffled)):
                     yield (torch.FloatTensor(chunk_filled_shuffled.iloc[j].values),
                           torch.FloatTensor(mask_df_shuffled.iloc[j].values),
                           torch.FloatTensor(c_shuffled[j]))

            except pd.errors.EmptyDataError:
                 # Handle case where chunk is empty (e.g., end of file)
                 print(f"Info: Reached end of file at chunk {chunk_idx}.")
                 break # Stop iteration

            except Exception as e:
                 print(f"Error processing chunk {chunk_idx}: {e}")
                 # Optionally skip this chunk or handle the error
                 # For training, you might want to skip problematic chunks or raise an error
                 continue # Skip to the next chunk


# --- preprocess_chunk 함수 ---
# This function is used within __iter__ but defined outside the class (e.g., cell ce812be1).
# It takes a DataFrame (potentially with NaNs) and returns filled data, mask, and conditional tensors.
# Ensure this function is defined before executing the cell that defines LikertDataset.

# --- introduce_missing_in_chunk function ---
# This function is used within __iter__ but defined outside the class (in the same cell 7df5ad7e).
# It takes a chunk DataFrame and missingness config and returns a DataFrame with artificial NaNs.
# Its definition is above this class definition within the same cell.

#PreLoop

In [56]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
# from tqdm.auto import tqdm # Remove tqdm
import math # Import math for cyclical beta

# Set device - prioritize CUDA, then default to CPU
# Removed TPU specific device check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Ensure required variables from hyperparameters are defined
# Check for the variable names defined in the hyperparameters cell (QT_6WTgCfUN3)
required_hyperparameters = ['FILE_PATH', 'CHUNK_SIZE', 'MISSINGNESS_CONFIG', 'all_columns',
                            'NUM_ITEMS', 'EMBEDDING_DIM', 'C_DIM', 'LATENT_DIM',
                            'BETA_WARMUP_EPOCHS', 'BETA_START', 'BETA_END',
                            'LEARNING_RATE', 'NUM_EPOCHS', 'BATCH_SIZE', 'NUM_RATINGS',
                            'EARLY_STOPPING_PATIENCE', 'ENCODER_HIDDEN_DIMS', 'DECODER_HIDDEN_DIMS']



for hp in required_hyperparameters:
    if hp not in globals():
        # Provide a more specific error message if the hyperparameter cell wasn't executed
        raise NameError(f"필수 하이퍼파라미터 '{hp}'가 정의되지 않았습니다. 하이퍼파라미터 셀(QT_6WTgCfUN3)을 먼저 실행했는지 확인하세요.")



# Initialize Dataset and DataLoader
# Use the modified LikertDataset and pass the missingness config for TRAINING
train_dataset = LikertDataset(
    FILE_PATH,
    chunk_size=CHUNK_SIZE,
    all_columns=all_columns, # Pass the all_columns list here
    mcar_cols=mcar_cols, # Pass the allocated columns
    mnar_cols=mnar_cols,
    mar_not_missing_pairs=mar_not_missing_pairs,
    mcar_total_missing_rate=MISSINGNESS_CONFIG['mcar'][1], # Pass the target rates
    mnar_total_missing_rate=MISSINGNESS_CONFIG['mnar'][1],
    mar_total_missing_rate=MISSINGNESS_CONFIG['mar'][1]
)

# Initialize LikertDataset for VALIDATION - WITHOUT artificial missingness config
# This dataset will read the original file and preprocess it (fill NaNs, create mask)
# but will *not* introduce additional artificial NaNs via introduce_missing_in_chunk.
# This evaluates performance on the original data distribution (including its natural missingness).
# Ensure preprocess_chunk is defined before LikertDataset instantiation if called in __init__ (it's not)
# but it is called in __iter__ which is fine.
val_dataset = LikertDataset(
    FILE_PATH,
    chunk_size=CHUNK_SIZE, # Use same chunk size for consistency
    all_columns=all_columns, # Pass the all_columns list
    # Do NOT pass mcar_cols, mnar_cols, etc. to disable artificial missingness generation
)



# With IterableDataset, DataLoader does not have a __len__ in the same way as Map-style
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)


# Initialize model, optimizer
# Pass the hidden layer dimensions to the CVAE constructor
model = CVAE(num_items=NUM_ITEMS,
             embedding_dim=EMBEDDING_DIM, # Note: Assuming embedding_dim == NUM_RATINGS
             c_dim=C_DIM,
             latent_dim=LATENT_DIM,
             encoder_hidden_dims=ENCODER_HIDDEN_DIMS, # Pass encoder hidden dims
             decoder_hidden_dims=DECODER_HIDDEN_DIMS, # Pass decoder hidden dims
             num_ratings=NUM_RATINGS
             ).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training Loop
global_step = 0

# --- Early Stopping Variables ---
best_val_loss = float('inf')
epochs_without_improvement = 0
# Note: EARLY_STOPPING_PATIENCE should be defined in Hyperparameters cell (QT_6WTgCfUN3)
# If EARLY_STOPPING_START_EPOCH is used, define it in Hyperparameters cell too.
# EARLY_STOPPING_START_EPOCH = 0 # Example: Start checking from epoch 0
# --- End Early Stopping Variables ---




Using device: cpu
Dataset initialized with 1015341 total rows (approx).
Missingness config: MCAR(0, 0.3), MNAR(50, 0.3), MAR(0, 0.3)
Dataset initialized with 1015341 total rows (approx).
Missingness config: MCAR(0, 0.0), MNAR(0, 0.0), MAR(0, 0.0)


#Loop

In [ ]:
print("Starting training...")


for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    train_reconstruction_loss = 0.0
    train_kl_loss = 0.0
    num_train_batches = 0

    if epoch < BETA_FIX_EPOCHS:
        beta = BETA_START
    else:
        # Calculate current beta using linear warmup
        if epoch < BETA_WARMUP_EPOCHS:
            beta = BETA_START + (BETA_END - BETA_START) * ((epoch - BETA_FIX_EPOCHS) / BETA_WARMUP_EPOCHS)
        else:
            beta = BETA_END


    # Wrap train_loader with tqdm for progress bar if desired (currently removed)
    # train_loop = tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} Training") # Removed tqdm

    for i, (x, mask, c) in enumerate(train_loader):
        x, mask, c = x.to(device), mask.to(device), c.to(device)

        optimizer.zero_grad()

        # Forward pass
        logits, mu, logvar = model(x, c)

        # Calculate losses
        reconstruction_loss = ordinal_cross_entropy_loss(logits, x, mask)
        kl_loss = kl_divergence_loss(mu, logvar, free_bits=FREE_BITS)
        loss = combined_loss(reconstruction_loss, kl_loss, beta)


        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_reconstruction_loss += reconstruction_loss.item()
        train_kl_loss += kl_loss.item() * beta # Store the beta-weighted KL for reporting

        global_step += 1
        num_train_batches += 1

        # Print batch progress periodically
        if (i + 1) % 1000 == 0: # Print every 1000 batches
             print(f"  Epoch [{epoch+1}/{NUM_EPOCHS}], Batch [{i+1}], Loss: {loss.item():.4f}, Rec Loss: {reconstruction_loss.item():.4f}, KL Loss (weighted): {kl_loss.item() * beta:.4f}, Beta: {beta:.4f}")



    # Average training loss for the epoch - Divide by number of batches processed
    avg_train_loss = train_loss / num_train_batches if num_train_batches > 0 else 0.0
    avg_train_reconstruction_loss = train_reconstruction_loss / num_train_batches if num_train_batches > 0 else 0.0
    avg_train_kl_loss = train_kl_loss / num_train_batches if num_train_batches > 0 else 0.0


    # --- Validation Step (Re-integrated for monitoring) ---
    model.eval() # Set model to evaluation mode
    val_loss = 0.0
    val_reconstruction_loss = 0.0
    val_kl_loss = 0.0
    # Metrics calculation is complex with IterableDataset and might be slow.
    # Let's focus on loss for per-epoch validation monitoring.
    # Full metrics evaluation is best done using the separate evaluation cells on df_original_eval.
    num_val_batches = 0


    with torch.no_grad(): # Disable gradient calculation for validation
        for i, (x_val, mask_val, c_val) in enumerate(val_loader):
            x_val, mask_val, c_val = x_val.to(device), mask_val.to(device), c_val.to(device)

            # Forward pass on validation data
            logits_val, mu_val, logvar_val = model(x_val, c_val)

            # Calculate validation losses
            # Use the mask from preprocess_chunk on the original data
            reconstruction_loss_val = ordinal_cross_entropy_loss(logits_val, x_val, mask_val)
            kl_loss_val = kl_divergence_loss(mu_val, logvar_val, free_bits=FREE_BITS)
            loss_val = combined_loss(reconstruction_loss_val, kl_loss_val, beta) # Use current beta for consistency

            val_loss += loss_val.item()
            val_reconstruction_loss += reconstruction_loss_val.item()
            val_kl_loss += kl_loss_val.item() * beta

            num_val_batches += 1

    # Average validation loss for the epoch
    avg_val_loss = val_loss / num_val_batches if num_val_batches > 0 else 0.0
    avg_val_reconstruction_loss = val_reconstruction_loss / num_val_batches if num_val_batches > 0 else 0.0
    avg_val_kl_loss = val_kl_loss / num_val_batches if num_val_batches > 0 else 0.0


    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Summary:")
    print(f"  Train Loss: {avg_train_loss:.4f}, Rec Loss: {avg_train_reconstruction_loss:.4f}, KL Loss (weighted): {avg_train_kl_loss:.4f}, Beta: {beta:.4f}")
    print(f"  Val Loss: {avg_val_loss:.4f}, Rec Loss: {avg_val_reconstruction_loss:.4f}, KL Loss (weighted): {avg_val_kl_loss:.4f}") # Print validation losses



    # --- Early Stopping Check ---
    # Apply early stopping check after validation
    # Ensure EARLY_STOPPING_PATIENCE is defined in Hyperparameters cell
    if epoch >= EARLY_STOPPING_START_EPOCH: # Uncomment if using start epoch
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
            # --- Optional: Save the best model ---
            # Define a model save path, e.g., BEST_MODEL_PATH = '/tmp/best_cvae_model.pth'
            # torch.save(model.state_dict(), BEST_MODEL_PATH)
            print("  Validation loss improved. Resetting early stopping counter.")
            # --- End Optional Save ---
        else:
            epochs_without_improvement += 1
            print(f"  Validation loss did not improve for {epochs_without_improvement} epochs.")

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break # Break the training loop
        # --- End Early Stopping Check ---


print("Training finished.")

# After training, you can load the best model (if saved) and run final evaluation
# Example:
# try:
#     model.load_state_dict(torch.load(BEST_MODEL_PATH))
#     print("Loaded best model for final evaluation.")
# except FileNotFoundError:
#     print("No best model saved, using the last trained model.")

# Final evaluation on the validation set (or a dedicated test set) using separate cells

Starting training...
  Epoch [1/100], Batch [1000], Loss: 1.5263, Rec Loss: 1.5205, KL Loss (weighted): 0.0058, Beta: 0.0001
  Epoch [1/100], Batch [2000], Loss: 1.4838, Rec Loss: 1.4681, KL Loss (weighted): 0.0157, Beta: 0.0001
  Epoch [1/100], Batch [3000], Loss: 1.3748, Rec Loss: 1.3442, KL Loss (weighted): 0.0307, Beta: 0.0001
  Epoch [1/100], Batch [4000], Loss: 1.3867, Rec Loss: 1.3407, KL Loss (weighted): 0.0460, Beta: 0.0001
  Epoch [1/100], Batch [5000], Loss: 1.3417, Rec Loss: 1.2937, KL Loss (weighted): 0.0480, Beta: 0.0001
  Epoch [1/100], Batch [6000], Loss: 1.3764, Rec Loss: 1.3300, KL Loss (weighted): 0.0464, Beta: 0.0001
  Epoch [1/100], Batch [7000], Loss: 1.3491, Rec Loss: 1.2998, KL Loss (weighted): 0.0493, Beta: 0.0001
Epoch [1/100] Summary:
  Train Loss: 1.4112, Rec Loss: 1.3784, KL Loss (weighted): 0.0328, Beta: 0.0001
  Val Loss: 1.3189, Rec Loss: 1.2667, KL Loss (weighted): 0.0521
  Epoch [2/100], Batch [1000], Loss: 1.2711, Rec Loss: 1.2053, KL Loss (weighted):

#Save

In [ ]:
import torch
import os

model_save_path = '/content/drive/MyDrive/Projects/CVAE_MI/model_08072.pt'

# 저장할 디렉토리가 없으면 생성
save_dir = os.path.dirname(model_save_path)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"디렉토리 생성됨: {save_dir}")

# 모델의 state_dict 저장
try:
    torch.save(model.state_dict(), model_save_path)
    print(f"모델이 다음 경로에 저장되었습니다: {model_save_path}")
except Exception as e:
    print(f"모델 저장 중 오류가 발생했습니다: {e}")

#EVAL: CallModel

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

model_save_path = '/content/drive/MyDrive/Projects/CVAE_MI/model_08072.pt' # Ensure this path is correct

# 'your_folder/your_file.csv'를 Google Drive에 있는 실제 파일 경로로 바꾸세요.
file_path = '/content/drive/MyDrive/Projects/CVAE_MI/big5_100K.csv'
try:
    df_original_eval = pd.read_csv(file_path) #, delimiter='\t')
    print("CSV 파일을 성공적으로 불러왔습니다:")
    #display(df.info())
    display(df_original_eval.describe())

except FileNotFoundError:
    print(f"오류: 파일을 찾을 수 없습니다. 경로를 확인하세요: {file_path}")
except Exception as e:
    print(f"파일을 읽는 중 오류가 발생했습니다: {e}")


#EVAL: IntroduceMissingValues

In [ ]:
def introduce_missing_values(df, column_name, missingness_type, target_percentage):
    """
    Introduces missing values into a specified column of a DataFrame based on different missingness patterns.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the column to introduce missing values into.
        missingness_type (str): The type of missingness ('negative_uni',
                               'positive_uni', 'bidirection', or 'random').
        target_percentage (float): The desired percentage of missing values to introduce (e.g., 0.10 for 10%).

    Returns:
        pd.DataFrame: The DataFrame with missing values introduced in the specified column.
    """
    # Create a copy to avoid modifying the original DataFrame directly before applying NaNs
    df_modified = df.copy()

    # 1. Calculate the total number of values in the specified column
    total_values = df_modified[column_name].size

    # 2. Calculate the target number of missing values (10%) and round
    target_missing_count = int(round(total_values * target_percentage))

    # 3. Print the results
    # print(f"Total number of values in {column_name}: {total_values}")
    # print(f"Target number of missing values in {column_name} ({target_percentage*100:.0f}%): {target_missing_count}")

    # 4. Handle potential NaN values in the column before calculating probabilities or subsets by using `.dropna()`.
    # For 'random' missingness, we don't need to drop NaNs before selecting indices,
    # as we select from the entire column indices.
    if missingness_type != 'random':
        column_data_for_prob = df_modified[column_name].dropna()

        if column_data_for_prob.empty:
            print(f"No non-null values in {column_name} to introduce missingness.")
            return df_modified # Return df_modified as is

        # 5. Calculate the mean of the non-null values in the specified column.
        mean_value = column_data_for_prob.mean()

        # 6. Implement conditional logic based on the missingness_type:
        if missingness_type == 'negative_uni':
            # If missingness_type is 'negative_uni', select the subset of non-null values that are less than or equal to the mean.
            subset_for_missingness = column_data_for_prob[column_data_for_prob <= mean_value].copy()
            # Calculate the distance from the mean as mean_value - subset_value.
            distance_from_mean = mean_value - subset_for_missingness
        elif missingness_type == 'positive_uni':
            # If missingness_type is 'positive_uni', select the subset of non-null values that are greater than or equal to the mean.
            subset_for_missingness = column_data_for_prob[column_data_for_prob >= mean_value].copy()
            # Calculate the distance from the mean as subset_value - mean_value.
            distance_from_mean = subset_for_missingness - mean_value
        elif missingness_type == 'bidirection':
            # If missingness_type is 'bidirection', select all non-null values.
            subset_for_missingness = column_data_for_prob.copy()
            # Calculate the distance from the mean as the absolute difference: np.abs(subset_value - mean_value).
            distance_from_mean = np.abs(subset_for_missingness - mean_value)
        else:
            # If missingness_type is invalid, print an error message and return the original DataFrame.
            print(f"Error: Invalid missingness_type '{missingness_type}'. Use 'negative_uni', 'positive_uni', 'bidirection', or 'random'.")
            return df_modified # Return original df_modified if type is invalid

        # 7. If the selected subset for missingness is empty, print a message and return the original DataFrame.
        if subset_for_missingness.empty:
            print(f"No values in {column_name} match the criteria for missingness type '{missingness_type}' to introduce missingness.")
            return df_modified

        # 8. Calculate weights for each value in the selected subset based on the calculated distance from the mean.
        # Using distance + 1 to ensure positive weights and increasing probability with distance.
        weights = distance_from_mean + 1

        # 9. Normalize these weights to create selection probabilities for the subset.
        selection_probabilities = weights / weights.sum()

        # 10. Get the indices of the selected subset.
        subset_indices = subset_for_missingness.index

        # 11. Calculate the number of values to select from this subset to make missing, ensuring it does not exceed the size of the subset or the overall target missing count.
        num_to_select_from_subset = min(target_missing_count, len(subset_indices))

        if num_to_select_from_subset > 0:
            # 12. Use np.random.choice to randomly select the indices from the subset based on the calculated selection probabilities. Use replace=False to ensure unique indices.
            indices_to_make_missing = np.random.choice(
                subset_indices,
                size=num_to_select_from_subset,
                replace=False,  # Ensure unique indices are selected
                p=selection_probabilities  # Use calculated probabilities for selection
            )

            # 13. Introduce missing values (np.nan) into the specified column of the copied DataFrame at the selected indices using .loc[].
            df_modified.loc[indices_to_make_missing, column_name] = np.nan

            # 14. Print the number of indices selected to confirm the operation.
            # print(f"Attempted to introduce {num_to_select_from_subset} missing values in {column_name} based on criteria.")
            # print(f"Number of indices selected to make missing in {column_name}: {len(indices_to_make_missing)}")

        else:
            print(f"Target missing count is 0 or no values in {column_name} match the criteria for selection.")

    elif missingness_type == 'random':
         # If missingness_type is 'random', select indices randomly from the entire column
         # We need to ensure we don't try to make more values missing than available
         num_to_select_random = min(target_missing_count, total_values)

         if num_to_select_random > 0:
             # Get all indices in the column
             all_indices = df_modified[column_name].index

             # Randomly select indices to make missing
             indices_to_make_missing = np.random.choice(
                 all_indices,
                 size=num_to_select_random,
                 replace=False # Ensure unique indices are selected
             )

             # Introduce missing values (np.nan)
             df_modified.loc[indices_to_make_missing, column_name] = np.nan

            #  print(f"Attempted to introduce {num_to_select_random} random missing values in {column_name}.")
            #  print(f"Number of indices selected to make missing in {column_name}: {len(indices_to_make_missing)}")

         else:
             print(f"Target missing count is 0 or no values in {column_name} available to make missing.")


    # 15. Calculate the actual number of missing values in the column after introducing NaNs.
    actual_missing_count = df_modified[column_name].isnull().sum()

    # 16. Calculate the actual percentage of missing values and print it, formatted to two decimal places.
    actual_missing_percentage = (actual_missing_count / total_values) * 100
    # print(f"\nActual number of missing values in {column_name} after introduction: {actual_missing_count}")
    # print(f"Actual percentage of missing values in {column_name}: {actual_missing_percentage:.2f}%")

    # 17. Return the modified DataFrame.
    return df_modified

#EVAL: ImputeMissingValues

In [ ]:
# --- 결측치 복원 함수 ---
def impute_missing_values(model, data_tensor_encoded, mask_tensor, conditional_tensor, device, num_ratings):
    """
    model: 학습된 CVAE 모델
    data_tensor_encoded: 결측치 0으로 인코딩된 데이터 tensor (N, num_items) - 이미 임베딩을 거치지 않은 원본 데이터 형태
    mask_tensor: 마스크 tensor (N, num_items), 1 for observed, 0 for missing
    conditional_tensor: 조건 벡터 c tensor (N, c_dim)
    device: cpu or cuda
    num_ratings: 리커트 척도 레벨 수 (5, 7 등)

    반환: imputed tensor (N, num_items), int (1~num_ratings)
    """
    model.eval() # Set model to evaluation mode
    data_tensor_encoded = data_tensor_encoded.to(device)
    mask_tensor = mask_tensor.to(device)
    conditional_tensor = conditional_tensor.to(device)


    with torch.no_grad(): # Disable gradient calculation
        # Encode the input (data_tensor_encoded is the raw data before embedding)
        # The CVAE forward method handles embedding internally.
        # We need to call the forward method of the CVAE model, which takes x and c.
        # The forward method returns logits, mu, logvar.
        logits, mu, logvar = model(data_tensor_encoded, conditional_tensor)


        # For deterministic imputation, use the mean of the latent space (mu)
        # For stochastic imputation, sample from the latent space
        # Let's use deterministic imputation (mu) for evaluation consistency
        # To use mu for decoding, we need to call the decoder directly.
        # The decoder expects z and c.
        # decoded_outputs_logits = model.decoder(mu, conditional_tensor) # Shape (batch_size, num_items, num_ratings)


        # Get the predicted category (most probable Likert score) from the logits
        # The logits are already in the shape (N, num_items, num_ratings) from the model's forward pass
        # argmax over the category dimension (dim=2)
        # Predicted categories are 0-indexed (0 to num_ratings-1)
        predicted_categories = torch.argmax(logits, dim=2) # shape (N, num_items)


        # Convert 0-indexed categories back to 1-indexed Likert scores (1 to num_ratings)
        imputed_vals = predicted_categories + 1 # shape (N, num_items), values 1-num_ratings


        # Create the final imputed tensor:
        # Start with the original data tensor (which has 1-num_ratings for observed, 0 for missing)
        # We need to ensure data_tensor_encoded is on the CPU before using it as the base
        imputed_tensor_final = data_tensor_encoded.cpu().clone()

        # Identify locations that were originally missing (mask is 0)
        # missing_mask shape: (N, num_items) -> boolean (on the same device as mask_tensor)
        missing_mask = (mask_tensor.cpu() == 0) # Ensure mask is on CPU for indexing the CPU tensor


        # Replace the values at missing locations in the final tensor with the imputed_vals
        # Ensure imputed_vals is on CPU for indexing the CPU tensor
        imputed_tensor_final[missing_mask] = imputed_vals.cpu()[missing_mask]


    return imputed_tensor_final # Return tensor on CPU

#EVAL: Components

In [ ]:
import numpy as np
import pandas as pd
import torch # Ensure torch is imported

# Ensure the necessary functions and model are defined and available:
# preprocess_chunk (from ce812be1), impute_missing_values (from 7hD_sk-6ED3w),
# CVAE (from jfFhQgsf8UyC), and loaded_model (from xdzukH359run)

# Check if required components are available
required_components = ['preprocess_chunk', 'impute_missing_values', 'CVAE', 'model', 'NUM_RATINGS', 'NUM_ITEMS', 'EMBEDDING_DIM', 'C_DIM', 'LATENT_DIM', 'ENCODER_HIDDEN_DIMS', 'DECODER_HIDDEN_DIMS', 'model_save_path'] # Added model_save_path for clarity
for component in required_components:
    if component not in globals():
        print(f"Error: Required component '{component}' is not defined. Please ensure all necessary code cells have been run.")
        # For this task, we must stop if core components are missing.
        raise NameError(f"Required component '{component}' is not defined.")


# Re-initialize the model structure (it will be loaded with saved weights later)
# Use the hyperparameters defined earlier
loaded_model = CVAE(num_items=NUM_ITEMS,
                    embedding_dim=EMBEDDING_DIM,
                    c_dim=C_DIM,
                    latent_dim=LATENT_DIM,
                    encoder_hidden_dims=ENCODER_HIDDEN_DIMS,
                    decoder_hidden_dims=DECODER_HIDDEN_DIMS,
                    num_ratings=NUM_RATINGS
                    )

# Load the saved model state dictionary

try:
    # Load the model state dictionary, mapping it to the CPU
    loaded_model.load_state_dict(torch.load(model_save_path, map_location=torch.device('cpu')))
    print(f"Model state dictionary loaded successfully from {model_save_path} and mapped to CPU.")
except FileNotFoundError:
    print(f"Error: Model file not found at {model_save_path}. Please check the path and ensure the model was saved.")
    raise
except Exception as e:
    print(f"Error loading model state dictionary: {e}")
    raise


# Set the device
# Ensure the device is correctly set based on availability after loading
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Ensure the loaded_model is on the correct device and in evaluation mode
loaded_model.to(device)
loaded_model.eval()


# Load the original data for evaluation (assuming it's the same file used for training)
# Load the entire dataset into a DataFrame for evaluation purposes.
# This might be memory intensive for very large files.
# If memory is an issue, evaluation might need to be done chunk by chunk as well.
# Ensure FILE_PATH is defined (it should be from the hyperparameters cell)
if 'FILE_PATH' not in globals():
     raise NameError("FILE_PATH variable is not defined. Please run the hyperparameters cell.")

try:
    # Use df_original_eval which was loaded in cell 8qFj8HCdL7bk or PS5pgwlLOTLa
    # Assuming df_original_eval is already loaded and available from previous cells.
    # If not, you would need to load it here. Let's assume it's available.
    if 'df_original_eval' not in globals():
        # If df_original_eval was filtered in PS5pgwlLOTLa and renamed to df_original_filtered, use that.
        # Or reload the original file if needed.
        # For evaluation, it's often better to load the original data here explicitly if needed.
        print(f"Loading original data for evaluation from {FILE_PATH}...")
        df_original_eval = pd.read_csv(FILE_PATH)
        print(f"Original data loaded for evaluation. Shape: {df_original_eval.shape}")

    all_columns_eval = df_original_eval.columns.tolist()
    # Check if column names match the global all_columns if that was used elsewhere
    if 'all_columns' in globals() and all_columns_eval != all_columns:
         print("Warning: Column names in evaluation data differ from training data.")
         # Decide how to handle this - for simplicity, assume they are the same.
         # Or update all_columns if necessary. Let's proceed assuming they are compatible.
         all_columns = all_columns_eval # Update all_columns for consistency


except FileNotFoundError:
    print(f"Error: Original data file not found for evaluation at {FILE_PATH}. Please check the path.")
    raise
except Exception as e:
    print(f"Error loading original data for evaluation: {e}")
    raise


# Define the list of columns to introduce missingness and evaluate
columns_to_process_eval = all_columns_eval # Use columns from the loaded evaluation data


# --- Evaluation Loop Setup ---
result = []

print("Starting evaluation across different missingness scenarios...")

# The actual evaluation loop will be in the next cell

#EVAL: PreProcess

In [ ]:
# (df == 0) 각 요소가 0인지 여부 Bool DataFrame을 생성
# .any(axis=1) 각 행에 대해 열에 0이 있는지 확인
rows_with_zero = (df_original_eval == 0).any(axis=1)

# 0을 포함하지 않는 행만 선택, 새로운 DataFrame을 생성
# df = df[~rows_with_zero].copy() # Fix: Changed 'df' to 'df_original_eval'
df_filtered = df_original_eval[~rows_with_zero].copy()


# print("원본 DataFrame의 행 수:", len(df)) # Fix: Original df is df_original_eval
print("원본 DataFrame의 행 수:", len(df_original_eval))
# print("0을 포함하는 행을 삭제한 후의 DataFrame 행 수:", len(df)) # Fix: Changed 'df' to 'df_filtered'
print("0을 포함하는 행을 삭제한 후의 DataFrame 행 수:", len(df_filtered))

# display(df.describe()) # Fix: Changed 'df' to 'df_filtered'
display(df_filtered.describe())

# df_original = df.copy() # Fix: Changed 'df' to 'df_filtered' and variable name
df_original_filtered = df_filtered.copy()

#Evaluation

In [ ]:
# Ensure the necessary variables and functions are defined and available from the previous cells:
# df_original_eval, columns_to_process_eval, loaded_model, device, NUM_RATINGS,
# introduce_missing_values (from this cell), preprocess_chunk (from ce812be1),
# impute_missing_values (from 7hD_sk-6ED3w), result (list from Z1l3izmy4phC)


# Check if required components are available
required_components_eval_loop = ['df_original_eval', 'columns_to_process_eval', 'loaded_model', 'device', 'NUM_RATINGS',
                                'introduce_missing_values', 'preprocess_chunk', 'impute_missing_values', 'result']

for component in required_components_eval_loop:
    if component not in globals():
        print(f"Error: Required component '{component}' is not defined. Please ensure all necessary code cells have been run.")
        raise NameError(f"Required component '{component}' is not defined.")


# Loop through missingness types
for mis_type in ['random', 'negative_uni', 'positive_uni', 'bidirection']:
    # Loop through missingness rates
    for mis_rate in [0.10, 0.30, 0.50]:
        print(f"\nProcessing scenario: mis_type='{mis_type}', mis_rate={mis_rate:.2f}")

        # Start with a fresh copy of the original dataframe for each scenario
        df_scenario = df_original_eval.copy()

        # Introduce missing values column by column for the current mis_type and mis_rate
        # The introduce_missing_values function expects a target_percentage per column.
        # Note: introduce_missing_values introduces np.nan
        for col in columns_to_process_eval:
             df_scenario = introduce_missing_values(df_scenario, col, mis_type, mis_rate)

        # --- Imputation Step ---
        # Preprocess the data with introduced missing values
        # preprocess_chunk returns filled data (0 for missing), mask, and conditional vector c
        # It operates on a DataFrame and returns NumPy arrays/DataFrame
        # Note: df_scenario_filled has 0s where df_scenario had NaNs. mask_df_scenario has 0s where df_scenario had NaNs.
        df_scenario_filled, mask_df_scenario, c_scenario = preprocess_chunk(df_scenario)

        # Convert preprocessed data and mask to PyTorch tensors
        # Ensure data is float or long as required by the model (data is used for embedding, so needs to be long)
        # The original Likert values are integers 1-5. preprocess_chunk fills NaNs with 0.
        # So the tensor should contain 0s and 1-5s. The embedding layer expects long.
        data_tensor_scenario = torch.LongTensor(df_scenario_filled.values)
        mask_tensor_scenario = torch.FloatTensor(mask_df_scenario.values)
        c_tensor_scenario = torch.FloatTensor(c_scenario)


        # Perform imputation using the loaded model
        # The impute_missing_values function now takes the data tensor (with 0s for missing), mask, and conditional tensor
        imputed_tensor = impute_missing_values(loaded_model, data_tensor_scenario, mask_tensor_scenario, c_tensor_scenario, device, NUM_RATINGS)

        # Convert imputed tensor back to a pandas DataFrame
        # The imputed_tensor contains values 1-NUM_RATINGS (e.g., 1-5) at both original and imputed locations
        df_imputed_scenario = pd.DataFrame(imputed_tensor.numpy(), columns=columns_to_process_eval, index=df_scenario.index)


        # --- Evaluation Step ---
        # Identify the locations where missing values were INTRODUCED by introduce_missing_values (these are NaNs in df_scenario)
        introduced_missing_mask = df_scenario.isna()

        # Use df_original_eval to get the TRUE values at these introduced missing locations
        # Align df_original_eval with df_scenario's index
        df_original_aligned_for_eval = df_original_eval.loc[df_scenario.index]

        # Extract original values for the cells that are NOW missing in df_scenario (these are the ground truth)
        # Use the boolean mask to directly index the original dataframe
        original_missing_values = df_original_aligned_for_eval.values[introduced_missing_mask.values]

        # Extract imputed values for the cells that were missing in df_scenario
        # Use the SAME boolean mask to directly index the imputed dataframe
        imputed_values_for_missing = df_imputed_scenario.values[introduced_missing_mask.values]


        # --- Filter out any potential NaNs before converting to int ---
        # This is a safety measure to handle cases where indexing might not perfectly align
        # or if introduce_missing_values left some NaNs in df_original_aligned_for_eval unexpectedly
        # (though based on its logic, it shouldn't).
        # We only want to evaluate on locations where BOTH original and imputed values are non-NaN numbers.

        # Create a boolean mask where both original and imputed values are NOT NaN
        valid_evaluation_mask = (~np.isnan(original_missing_values)) & (~np.isnan(imputed_values_for_missing))

        # Apply this mask to filter the arrays
        original_missing_values_filtered = original_missing_values[valid_evaluation_mask]
        imputed_values_for_missing_filtered = imputed_values_for_missing[valid_evaluation_mask]

        # --- End Filtering ---


        # Check if there are actually any missing values introduced and imputed in this scenario AFTER filtering
        if len(original_missing_values_filtered) == 0:
            print(f"No valid missing values found for evaluation after filtering NaNs in scenario mis_type='{mis_type}', mis_rate={mis_rate:.2f}. Skipping metrics calculation.")
            result.append({
                'mis_type': mis_type,
                'mis_rate': mis_rate,
                'accuracy': np.nan,
                'mae': np.nan,
                'rmse': np.nan
            })
            continue

        # The number of original missing values and imputed values for those locations should now match
        # because we are using the same mask to extract from aligned dataframes.
        # The previous error check `if len(original_missing_values) != len(imputed_values_for_missing):`
        # is no longer strictly necessary with this extraction method, but can be kept as a sanity check.
        # Let's check the filtered arrays instead.
        if len(original_missing_values_filtered) != len(imputed_values_for_missing_filtered):
             print(f"Sanity Check Failed: Mismatch in number of original missing ({len(original_missing_values_filtered)}) and imputed ({len(imputed_values_for_missing_filtered)}) values AFTER FILTERING for scenario mis_type='{mis_type}', mis_rate={mis_rate:.2f}. This should not happen with direct indexing. Skipping metrics calculation.")
             result.append({
                 'mis_type': mis_type,
                 'mis_rate': mis_rate,
                 'accuracy': np.nan,
                 'mae': np.nan,
                 'rmse': np.nan
             })
             continue


        # Convert to numpy arrays for easier calculation and ensure they are integers
        # Now converting the filtered arrays
        original_missing_values_int = original_missing_values_filtered.astype(int)
        imputed_values_for_missing_int = imputed_values_for_missing_filtered.astype(int)

        # Calculate Imputation Accuracy
        accuracy = np.mean(original_missing_values_int == imputed_values_for_missing_int)

        # Calculate Mean Absolute Error (MAE)
        mae = np.mean(np.abs(original_missing_values_int - imputed_values_for_missing_int))

        # Calculate Root Mean Squared Error (RMSE)
        rmse = np.sqrt(np.mean((original_missing_values_int - imputed_values_for_missing_int)**2))

        # Append the calculated metrics to the result list
        result.append({
            'mis_type': mis_type,
            'mis_rate': mis_rate,
            'accuracy': accuracy,
            'mae': mae,
            'rmse': rmse
        })


print("\n--- Evaluation Complete ---")

# Convert results to a DataFrame for better readability
results_df = pd.DataFrame(result)

# Display the results summary
print("\nEvaluation Results Summary:")
display(results_df)